# Análise das Transcrições das Lives do Frei Gilson - Quaresma 2025

## Introdução

**Autor:** *Bruno Conterato*

**Objetivo:** *Analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025 utilizando técnicas modernas de processamento de linguagem natural (NLP).*

---

## Descrição
Este notebook tem como objetivo analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Etapas de Pré-processamento
Este notebook descreve as etapas de pré-processamento necessárias para analisar as transcrições das lives do Frei Gilson durante a Quaresma de 2025.

---

## Necessidades de Processamento
- **Identificar o ponto de divisão** entre as reflexões do Terço e as reflexões do Dia.
- **Separar os textos** em reflexões do Terço e reflexões do Dia.
- **Remover as seções** marcadas com a tag `[Música]`.
- **Identificar e remover orações oficiais** da Igreja Católica.
- **Extrair e documentar músicas cantadas**, incluindo o nome e o autor de cada música.
- **Considerar o momento do Rosário** em que cada ensinamento foi apresentado, relacionando-o ao terço e ao mistério meditado naquele momento.

## 1.0. Importação de Bibliotecas


In [1]:
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# MODEL = "deepseek-r1:8b"
# MODEL = "deepseek-r1:14b"
# MODEL = "deepseek-r1:8b-llama-distill-q4_K_M"
# MODEL = "deepseek-r1:8b-llama-distill-q8_0" # Trouxe só 1 versículo, e em Inglês
# MODEL = "deepseek-r1:14b-qwen-distill-q4_K_M" # Resumo ficou interessante. Traz versículos bíblicos.
# MODEL = "gemma3:12b"
# MODEL = "gemma3:27b-it-qat" # [18 GB] [Lento]
# MODEL = "gemma3:12b-it-q4_K_M"  # 8.1 GB
# MODEL = 'gemma3:12b-it-qat'   # [8.9GB]
# MODEL = "gemma3:12b-it-q8_0"  # [13 GB] # Identifica 2 versículos, 1 sem transcrição
# MODEL = "phi4:14b"
# MODEL = "mistral-small3.1:24b"
# MODEL = "DarkIdol-Llama-3.1-8B-Instruct-1.2-Uncensored:latest"
# MODEL = "llama3.1:8b"
# MODEL = "llama3.1:8b-instruct-q5_K_M"            # Bom resultado para versículos bíblicos
# MODEL = "mistral:7b"
MODEL = "mistral:7b-instruct"
# MODEL = "openchat:7b"
# MODEL = "qwen2.5:7b-instruct"
# MODEL = "qwen3:8b"  # [5.2GB]    #Identificou corretamente 3 versículos bíblicos
# MODEL = "qwen3:14b" # [9.3GB]   # Identificou trechos de 2 versículos bíblicos
# MODEL = "nous-hermes2:10.7b"    # [6.1GB]

# List of Models: https://ai.google.dev/gemini-api/docs/models
# MODEL = "gemini-2.0-flash"

# Providers: azure_openai, deepseek, bedrock, openai, fireworks, xai, mistralai,
# ollama, google_anthropic_vertex, cohere, google_vertexai, perplexity,
# azure_ai, google_genai, ibm, bedrock_converse, groq, together, anthropic, huggingface
MODEL_PROVIDER = "ollama"
# MODEL_PROVIDER = "google_genai"

## 2.0. Processamento de texto

In [3]:
def remove_starting_tabs(text):
    """
    Remove starting tabs from the text.
    """
    lines = text.split("\n")
    cleaned_lines = [line.lstrip("\t") for line in lines]
    return "\n".join(cleaned_lines)

In [4]:
# system_message = f"""
#     Você é um especialista na fé católica.
#     Você receberá um texto que é uma transcrição de um vídeo do YouTube.

#     O vídeo é Santo Rosário, rezado diariamente ao vivo pelo Frei Gilson durante a um dia da quaresma do ano de 2025.
#     Durante o vídeo, o Frei Gilson reza o Santo Rosário e faz uma série de reflexões e ensinamentos sobre a fé católica.

#     O vídeo é dividido em duas partes bem distintas:
#     1. A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
#     2. A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

#     Você vai se atentar a informações prestadas apenas durante a primeira parte do vídeo (o Santo Rosário),
#     e vai recuperar essas informações devidamente formatadas e organizadas,
#     a fim de que o usuário possa ter um resumo de todo ensinamento que foi transmitido.
    
#     Você vai trazer somente informações que estejam presentes na primeira parte do vídeo: o Santo Rosário.
#     Você não pode trazer informações que estejam presentes apenas na segunda parte do vídeo (a reflexão),
#     a menos que também o Santo Rosário do dia traga esse conhecimento.
    
#     Você não precisa explicar o funcionamento do Santo Rosário, 
#     a menos que essa informação seja relevante para algum ensinamento que você trouxer.
    
#     Você não precisa trazer transcricões de oracões católicas conhecidas, como o Pai Nosso, a Ave Maria ou o Credo.
    
#     Ignore as marcações de tempo presentes na transcrição.
#     Não invente informações, não faça suposições, não faça inferências, somente traga informações presentes na transcrição.
    
#     Você vai responder apenas com um relatório markdown, exatamente com as mesmas seções e formatações que estão no exemplo abaixo:
    
#     ```markdown
#     # Relatório do Santo Rosário
    
#     ## Temática principal
    
#     - <Neste Santo Rosário o Frei Gilson escolhe uma temática chave sobre a vida e a fé cristã, e a aborda com ensinamentos e reflexões entre as orações. Identifique a temática principal e traga-a aqui.>
#     <2 a 3 parágrafos de informações e ensinamentos sobre a temática principal dos ensinamentos trazidos junto a este Santo Rosário>
    
#     ## Temáticas secundárias
    
#     <O Frei Gilson também aborda temáticas secundárias entre as orações do Santo Rosário. Você vai identificá-las e trazer nas seções abaixo, começando pela mais relevante>
    
#     ### <temática secundária 1/2>
    
#     <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária 1>
    
#     ### <temática secundária 2/2>
    
#     <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária 2>
    
#     ...
    
#     ### <temática secundária N>
    
#     <1 a 2 parágrafos de informações e ensinamentos que foram transmitidos sobre a temática secundária N (No mínimo 2 e no máximo 5 temáticas secundárias)>
    
#     ## Versículos citados na transcrição
    
#     <Identifique cuidadosamente todos os versículos que foram citados na transmissão do Santo Rosário>
#     <Você vai trazer a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson>
#     <Se houver necessidade, identifique o momento do Santo Rosário em que o versículo foi citado (Qual terço, qual mistério, etc.)>
#     <Você não vai deixar de fora nenhum versículo que tenha sido citado>
#     <Pode haver intervalos de versículos, como por exemplo: Salmo 1, 1-3>
    
#     Ex.:
    
#     - Se o versículo Livro N, A (ou A-B caso seja m intervalo de versículos) foi citado, você vai trazer:
    
#     (Livro N, A) ou (Livro N, A-B): <Transcrição integral do versículo>
    
#     Ensinamentos: <Parágrafo de ensinamentos que foram transmitidos por aquele versículo>
    
#     ## Músicas
    
#     <Lista de todas as músicas que foram citadas durante o Santo Rosário>
    
#     Formato:
    
#     <Nome da música> - <Artista>
#     <Descrição do ensinamento que foi transmitido sobre a música>
    
#     ## Eventos de Agenda
    
#     <Lista de todos os eventos de agenda que foram citados durante o Santo Rosário>
    
#     Formato.:
    
#     <Título> - <data e horário> - <local>
#     <Descrição do evento>
#     ```
    
#     Por favor, não traga seções vazias, nem seções que não estejam no exemplo acima.
#     Nã responda nada além do relatório markdown exatamente como o exemplo acima.
# """

In [5]:
# Aprimorado para LLMs menores pelo ChatGPT
system_message = """
    Você é um especialista na fé católica.  
    Receberá um **texto com a transcrição da oração do Santo Rosário**, rezado ao vivo pelo Frei Gilson durante um dia da Quaresma de 2025.

    A oração está dividida em duas partes:
    - A primeira parte é o Santo Rosário, onde Frei Gilson reza o Rosário com os fiéis.
    - A segunda parte é uma reflexão relacionada à fé católica e ao silêncio.

    ⚠️ **IMPORTANTE:**  
    Ignore qualquer parte da transcrição que seja da parte da **reflexão**.  
    Use **apenas o que for dito durante a oração do Rosário**.

    ---

    ### Sua missão é extrair e organizar as informações nos itens de 1 a 5 abaixo:

    ---

    ## 1. Temática principal

    - Identifique a principal ideia que Frei Gilson ensina enquanto reza o Rosário.
    - Resuma em até 3 parágrafos.
    - Use somente ensinamentos que ele falou **durante a oração do Santo Rosário**.

    ---

    ## 2. Temáticas secundárias

    - Identifique de 2 a 5 temas que Frei Gilson também falou **durante a oração do Santo ROsário**.
    - Para cada tema:
    - Dê um título claro (ex: *Perseverança na fé*).
    - Escreva 1 a 2 parágrafos com os ensinamentos relacionados.

    ---

    ## 3. Versículos da Bíblia

    Identifique todos os **versículos bíblicos citados durante a oração**.  
    Não se esqueça de nenhum versículo ou intervalo de versículos.
    Traga apenas versículos que foram citados durante a oração do Santo Rosário.
    Traga a localização do versículo, a transcrição integral do versículo e os ensinamentos que foram transmitidos por aquele versículo e pela narração do Frei Gilson.
    Para cada um:

    - Escreva assim:  
    `(Livro Capítulo, Versículo)`: <Transcrição integral do versículo> ou
    `(Livro Capítulo, Versículo-Início–Versículo-Fim)`: <Transcrição integral do intervalo de versículos>

    - Logo abaixo, escreva:
    **Ensinamentos:**: <Parágrafo ou frase que explique o que o Frei falou sobre esse versículo durante a oração>

    Se o versículo for relacionado a algum mistério do Santo Rosário (como `Segundo Mistério Doloroso` ou `Quinto Mistério Gozozo`), diga isso.

    ---

    ## 4. Músicas

    Se Frei Gilson mencionar músicas durante a oração:

    - Diga o nome da música e o artista, como neste exemplo:
    <Nome da música> - <Artista>: <contexto>  

    - Depois escreva o que Frei Gilson disse sobre a música.

    ---

    ## 5. Eventos de Agenda

    Se Frei Gilson mencionar **eventos** (missas, encontros, lives etc.):
    Traga todos os eventos citados durante a oração ou ao final do Santo Rosário.
    - Traga o nome do evento, a data e o local.
    - Depois escreva o que ele falou sobre esse evento.

    ---

    ## ⚠️ Regras finais:

    - NÃO copie orações conhecidas (Pai Nosso, Ave Maria, Credo etc.).
    - Não explique o rosário, exceto se dor necessário para compreender o respectivo ensinamento.
    - NÃO traga informações da reflexão final que ocorre após o rosário. Traga só o que é dito durante o Rosário.
    - NÃO invente nada. Só use o que estiver escrito.
    - NÃO deixe seções em branco. Se algo não aparecer, **remova a seção inteira**.
    ```
"""

## 3.0 Detecção de trechos da bíblia

In [26]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.output_parsers.enum import EnumOutputParser, Enum
from langchain_core.prompts import PromptTemplate

from typing import List
from langchain_core.documents import Document

# 0. Carregue a transcrição
transcription = """
Hoje refletimos sobre a importância de sermos humildes. Como está escrito: "Bem-aventurados os humildes, pois herdarão a terra".
Mais adiante, mencionou-se que devemos amar o próximo como a nós mesmos.
"""

# 1. Divida a transcrição
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
chunks = splitter.create_documents([transcription])

# 2. Carregue os embeddings e a base vetorial
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
db = Chroma(
    collection_name="biblia",
    persist_directory="../Bíblia VectorStore/biblia_vectorstore",
    embedding_function=embedding_model,
)


# 3. Defina a enum e o parser
class BinaryResponse(Enum):
    YES = "Sim"
    NO = "Não"
    
class SelectResponse(Enum):
    OPTION_1 = "1"
    OPTION_2 = "2"
    OPTION_3 = "3"

binary_parser = EnumOutputParser(enum=BinaryResponse)

select_parser = EnumOutputParser(enum=SelectResponse)

# 4. Prompt
template = """
Use o seguinte contexto da Bíblia para responder à pergunta.
Não invente nada. Se não estiver no contexto, responda com a opção Não.

Contexto:

{context}

Trecho analisado:
{query}

Este trecho pode conter uma passagem da Bíblia?

{format_instructions}

Não traga nada além de `Sim` ou `Não`. Não explique sua resposta.
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["query", "context"],
    partial_variables={"format_instructions": binary_parser.get_format_instructions()},
)

# 5. LLM local
# Funciona bem com os modelos instruct:
# - llama3.1:8b-instruct-q5_K_M
# - mistral:7b-instruct
llm = ChatOllama(model=MODEL)

# 6. Cadeia completa manual (RAG customizado)
retriever = db.as_retriever(search_kwargs={"k": 3})


def format_context(docs: List[Document], show_option=False):
    if show_option:
        return "\n\n".join(
            f"Excerpt {i+1}: {doc.metadata["book"]} {doc.metadata["chapter"]}:{doc.metadata["verse"]} {doc.page_content}"
            for (i, doc) in enumerate(docs)
        )
    return "\n\n".join(
        f"{doc.metadata["book"]} {doc.metadata["chapter"]}:{doc.metadata["verse"]} {doc.page_content}"
        for doc in docs
    )


rag_chain = (
    {
        "query": lambda x: x["query"],
        # "context": lambda x: format_context(retriever.get_relevant_documents(x["query"]))
        "context": lambda x: format_context(retriever.invoke(x["query"])),
    }
    | prompt
    | llm
    | binary_parser
)

# 7. Execute o RAG para cada bloco
results = []
for chunk in chunks:
    resposta = rag_chain.invoke({"query": chunk.page_content})
    results.append(
        {
            "trecho": chunk.page_content,
            "resposta": resposta.value,  # .value retorna "Sim" ou "Não"
        }
    )

# Exemplo de print
for r in results:
    print("Trecho:", r["trecho"])
    print("Contém referência bíblica?", r["resposta"])
    print("----")

Trecho: Hoje refletimos sobre a importância de sermos humildes. Como está escrito: "Bem-aventurados os humildes, pois herdarão a terra".
Mais adiante, mencionou-se que devemos amar o próximo como a nós mesmos.
Contém referência bíblica? Sim
----


In [ ]:
select_template = """
Utilize como contexto:

{number_context}

Qual desses versículos mais se assemelha ao seguinte trecho:

{query}

Responda somente um número de 1 a 3. Não explique sua resposta.NO

{format_instructions}
"""

select_prompt = PromptTemplate(
    template=select_template,
    input_variables=["query", "number_context"],
    partial_variables={"format_instructions": select_parser.get_format_instructions()},
)


def get_bible_passages(transcription: str):
    bible_passages = []
    chunks = splitter.create_documents([transcription])
    for chunk in chunks:
        context = retriever.invoke(chunk.page_content)
        # resposta = rag_chain.invoke({"query": chunk.page_content})
        rag_pipeline = prompt | llm | binary_parser
        response = rag_pipeline.invoke(
            {"query": chunk.page_content, "context": format_context(context)}
        )
        if response == BinaryResponse.YES:
            select_pipeline = select_prompt | llm | select_parser
            option_number = select_pipeline.invoke(
                {
                    "query": chunk.page_content,
                    "number_context": format_context(context, show_option=True),
                }
            )
            bible_passages.append(
                {
                    "book": context[int(option_number.value) - 1].metadata["book"],
                    "chapter": context[int(option_number.value) - 1].metadata[
                        "chapter"
                    ],
                    "verse": context[int(option_number.value) - 1].metadata["verse"],
                    "text": context[int(option_number.value) - 1].page_content,
                }
            )

    return bible_passages


get_bible_passages(transcription="E Deus criou o céu e a terra.")

E Deus criou o céu e a terra.
[Document(id='919ecaba-de8f-4caa-9dc1-20d7a985c796', metadata={'book': 'Gênesis', 'chapter': '1', 'line_number': 0, 'verse': '1'}, page_content='No princípio, Deus criou o céu e a terra.\n'), Document(id='ac85f5c1-bdf1-42b7-9046-097c64cea33f', metadata={'book': 'Deuteronômio', 'chapter': '10', 'line_number': 5200, 'verse': '14'}, page_content='Vê, ao Senhor, teu Deus, pertencem o céu e o céu dos céus, a terra e tudo o que nela se encontra.\n'), Document(id='796dca4a-ffa9-4327-a71b-9ec828dc87db', metadata={'book': 'Gênesis', 'chapter': '2', 'line_number': 34, 'verse': '4'}, page_content='Tal é a história da criação do céu e da terra. No tempo em que o Senhor Deus fez a terra e o céu,\n')]
BinaryResponse.YES
1


[{'book': 'Gênesis',
  'chapter': '1',
  'verse': '1',
  'text': 'No princípio, Deus criou o céu e a terra.\n'}]

## 4.0. Leitura dos Arquivos

In [10]:
import os
from urllib import response

from tqdm import tqdm
from langchain.chat_models import init_chat_model
import pprint

model = init_chat_model(MODEL, model_provider=MODEL_PROVIDER)

raw_folder = "../../data/raw/Santo Rosário | Quaresma 2025/Youtube to Text"
processed_folder = "../../data/processed/Santo Rosário | Quaresma 2025/Youtube to Text"
titulo_template = (
    "Santo Rosário | Quaresma 2025 | 03:40 | {ordem}° Dia | Live Ao vivo.txt"
)

print("Diretório atual:", os.getcwd())


def gerar_titulo_fonte(ordem):
    return titulo_template.format(ordem=ordem)

for i in tqdm(range(13, 41), desc="Processando arquivos", unit="arquivo"):
    titulo_source = gerar_titulo_fonte(str(i))
    arquivo = f"{raw_folder}/{titulo_source}"
    titulo_md = f"{titulo_source[:-3]}.md"

    if os.path.exists(arquivo):
        with open(arquivo, "r+", encoding="utf-8") as f:
            conteudo = f.read()
            if conteudo:
                print(f"\n\nArquivo: {titulo_source}")
                
                conteudo += "\n\nInstruções:\n\n" + remove_starting_tabs(system_message)

                messages = [
                    # {"role": "system", "content": remove_starting_tabs(system_message)},
                    {"role": "user", "content": conteudo},
                ]
                    
                # response = model.invoke(messages).content
                
                chunks = []
                for chunk in model.stream(messages):
                    chunks.append(chunk.text())
                    print(chunk.text(), end="", flush=True)
                response = "".join(chunks)
                
                
                # with open(
                #     f"{processed_folder}/rosario_{titulo_md}", "w+", encoding="utf-8"
                # ) as f:
                #     f.write(response)

            else:
                print(f"\n\nO arquivo {titulo_source} está vazio.")
    else:
        print(f"\n\nArquivo não encontrado: {arquivo}")
    
    break

Diretório atual: /home/bruno/Workspace/MariaGPT/src/Rosários Quaresma Frei Gilson 2025


Processando arquivos:   0%|          | 0/28 [00:00<?, ?arquivo/s]



Arquivo: Santo Rosário | Quaresma 2025 | 03:40 | 13° Dia | Live Ao vivo.txt
1. Temática principal: Devoção ao Santo Rosário e ao Coração de Jesus

   - Frei Gilson ensina a orar o Santo Rosário para meditar sobre os mistérios de Maria e de Jesus.
   - Ele also emphasizes the importance of contemplating the sufferings and joys of Jesus, as well as the virtues of Mary.
   - He also encourages the faithful to seek refuge in the heart of Jesus, asking for His mercy and protection.

  2. Temática secundária: Importância da Oração e Silêncio

   - Title: Importance of Prayer and Silence
   - Frei Gilson highlights that prayer is essential for communication with God and that silence allows us to listen to Him more attentively.

  3. Versículos da Bíblia

   - (Mateus 5, 44): "Porque, se o sol se leva a sua luz na esfera do céu e a lua e as estrelas se mostram no seu lugar, então também você deve fazer o seu pai que é no céu seja visto em você."
   - (1 Coríntios 2, 9): "Porque na carne não 

Processando arquivos:   0%|          | 0/28 [00:28<?, ?arquivo/s]
